# AB STOP/APO tandem-repeat screening

This notebook performs the two AB comparisons separately:

1. **STOP + APO vs DIVIDE**
2. **APO vs non-APO**

## Required input files

- `AB_BOTTOM_UP_TOP_DOWN.txt`: validated AB lineage output.
- `TR_CATALOG_RAW.tsv`: the raw tandem-repeat catalog generated by the project notebook [C_elegans_Lineage_Tandem_Repeats.ipynb](https://github.com/mfrbioengineering/TR/blob/main/C_elegans_Lineage_Tandem_Repeats.ipynb).

Run the TR-generation notebook first and provide its `TR_CATALOG_RAW.tsv` output here. This notebook does **not** rerun TRF and does not rebuild the catalog from other files. `TR_CATALOG_FINAL.tsv`, `TR_BINARY_MS.csv`, and `C_elegans_TR_MASTER_39098.csv` are not accepted as substitutes.

Upload both files when prompted. The notebook checks the lineage classifications and the TR catalog before scanning. Missing or inconsistent data cause an explicit error; no missing lineage state is inferred.

The scan uses the original rule: a window is promoted only when **every** monomer in one target class has identity **below 0.50** with **every** monomer in the comparison class. Results and checkpoints are saved separately for the two comparisons.

In [ ]:
from google.colab import files
from pathlib import Path
import io
import json
import numpy as np
import pandas as pd

uploaded = files.upload()
uploaded_names = list(uploaded)

def uploaded_file(required_name):
    matches = [name for name in uploaded_names if Path(name).name == required_name]
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one uploaded file named {required_name}; "
            f"found {len(matches)}. Uploaded files: {uploaded_names}"
        )
    return Path("/content") / matches[0]

LINEAGE_FILE = uploaded_file("AB_BOTTOM_UP_TOP_DOWN.txt")
TR_FILE = uploaded_file("TR_CATALOG_RAW.tsv")

print("Lineage file:", LINEAGE_FILE.name)
print("TR catalog:", TR_FILE.name)
print("TR catalog size:", f"{TR_FILE.stat().st_size:,}", "bytes")

In [ ]:
# Read and classify the validated AB lineage

lines = LINEAGE_FILE.read_text(
    encoding="utf-8-sig", errors="replace"
).splitlines()

def read_section(start_heading, next_heading=None):
    try:
        start = next(i for i, line in enumerate(lines) if line.strip() == start_heading)
    except StopIteration:
        raise ValueError(f"Missing lineage section: {start_heading}")

    if next_heading is None:
        end = len(lines)
    else:
        try:
            end = next(
                i for i in range(start + 1, len(lines))
                if lines[i].strip() == next_heading
            )
        except StopIteration:
            raise ValueError(f"Missing next section heading: {next_heading}")

    return {
        line.strip()[3:]
        for line in lines[start + 1:end]
        if line.strip().startswith("AB ")
    }

tree = read_section("-- ALBERO COMPLETO --", "========== STOP (391) ==========")
stop_nodes = read_section("========== STOP (391) ==========", "========== DIVIDE (486) ==========")
divide_section = read_section("========== DIVIDE (486) ==========", "========== APO INFERITE (92) ==========")
apo_inferred = read_section("========== APO INFERITE (92) ==========", "========== APO-HERM. NOMINATE (4) ==========")
apo_named = read_section("========== APO-HERM. NOMINATE (4) ==========")

apo_nodes = apo_inferred | apo_named
divide_nodes = divide_section & tree

if not tree:
    raise ValueError("The complete AB tree is empty.")

if stop_nodes & apo_nodes or stop_nodes & divide_nodes or apo_nodes & divide_nodes:
    raise ValueError("Overlapping lineage state classifications detected.")

classified = stop_nodes | apo_nodes | divide_nodes
unclassified = tree - classified
if unclassified:
    raise ValueError(
        f"{len(unclassified)} tree nodes have no explicit state. "
        f"Examples: {sorted(unclassified)[:10]}"
    )

if stop_nodes - tree or apo_nodes - tree:
    raise ValueError("STOP/APO entries occur outside the complete AB tree.")

roots = sorted(node for node in tree if len(node) == 5)
if len(roots) != 32:
    raise ValueError(f"Expected 32 AB roots; found {len(roots)}.")

print("Tree nodes (founder excluded):", len(tree))
print("AB roots:", len(roots))
print("DIVIDE nodes in the 32 subtrees:", len(divide_nodes))
print("STOP nodes:", len(stop_nodes))
print("Inferred APO:", len(apo_inferred))
print("Named APO:", len(apo_named))
print("Total APO:", len(apo_nodes))

In [ ]:
# Build ordered lineage positions and insert two virtual NULL positions after each APO

records = []

for root in roots:
    nodes = sorted(
        (node for node in tree if node.startswith(root)),
        key=lambda node: (len(node), node)
    )
    root_records = []

    for node in nodes:
        if node in divide_nodes:
            state, code = "DIVIDE", "0"
        elif node in stop_nodes:
            state, code = "STOP", "1"
        elif node in apo_nodes:
            state, code = "APO", "APO"
        else:
            raise ValueError(f"Unclassified node: {node}")

        root_records.append({
            "Root": root, "Node": node, "State": state,
            "Code": code, "Position_type": "BIOLOGICAL"
        })

        if state == "APO":
            for suffix in ("NULL1", "NULL2"):
                root_records.append({
                    "Root": root, "Node": f"{node}_{suffix}",
                    "State": "NULL", "Code": "NULL",
                    "Position_type": "VIRTUAL"
                })

    # Remove only terminal virtual NULL positions.
    last_biological = max(
        i for i, record in enumerate(root_records)
        if record["Position_type"] == "BIOLOGICAL"
    )
    root_records = root_records[:last_biological + 1]

    for position, record in enumerate(root_records, start=0):
        record["Position"] = position
        records.append(record)

lineage = pd.DataFrame(records)

biological = lineage[lineage["Position_type"] == "BIOLOGICAL"]
apo_total = int((biological["State"] == "APO").sum())
stop_total = int((biological["State"] == "STOP").sum())
divide_total = int((biological["State"] == "DIVIDE").sum())
null_total = int((lineage["Position_type"] == "VIRTUAL").sum())

assert len(roots) == 32
assert stop_total + apo_total + divide_total == len(biological)
assert apo_total == 96
assert stop_total == 391
assert divide_total == 455

root_lengths = lineage.groupby("Root").size()
minimum_root_length = int(root_lengths.min())

print("Biological positions:", len(biological))
print("DIVIDE:", divide_total)
print("STOP:", stop_total)
print("APO:", apo_total)
print("Internal virtual NULL positions:", null_total)
print("Minimum normalized root length:", minimum_root_length)
display(root_lengths.rename("Normalized_positions").to_frame())

In [ ]:
# Load and prepare the raw TR catalog

TR = pd.read_csv(TR_FILE, sep="\t", low_memory=False)

required_columns = {
    "Chromosome", "Start", "End", "Period", "Copy_Number",
    "Consensus_Size", "Percent_Matches", "Percent_Indels",
    "Score", "A", "C", "G", "T", "Entropy", "Consensus",
    "Repeat_Sequence"
}
missing_columns = required_columns - set(TR.columns)
if missing_columns:
    raise ValueError(f"TR_CATALOG_RAW.tsv is missing columns: {sorted(missing_columns)}")

TR["Period"] = pd.to_numeric(TR["Period"], errors="raise").astype(int)
TR["Original_length"] = TR["Repeat_Sequence"].str.len()
TR["Tail_length"] = TR["Original_length"] % TR["Period"]
TR["Complete_sequence"] = [
    sequence[:len(sequence) - tail] if tail else sequence
    for sequence, tail in zip(TR["Repeat_Sequence"], TR["Tail_length"])
]
TR["N_monomers"] = (
    TR["Complete_sequence"].str.len() // TR["Period"]
).astype(int)
TR["Monomers"] = [
    [sequence[i:i + period] for i in range(0, len(sequence), period)]
    for sequence, period in zip(TR["Complete_sequence"], TR["Period"])
]

# Keep TRs long enough for the shortest normalized AB root.
TR_FILTERED = TR[TR["N_monomers"] >= minimum_root_length].copy()

# Preserve the original short-period rule:
# Period >= 4 is retained; Period 2 or 3 is retained only if every monomer contains A.
short_period = TR_FILTERED[TR_FILTERED["Period"].isin([2, 3])].copy()
short_period["A_in_every_monomer"] = short_period["Monomers"].apply(
    lambda monomers: all("A" in monomer for monomer in monomers)
)

TR_FILTERED = pd.concat(
    [
        TR_FILTERED[TR_FILTERED["Period"] >= 4],
        short_period[short_period["A_in_every_monomer"]]
    ],
    ignore_index=True
)

print("Raw TRs:", len(TR))
print("TRs with incomplete tail removed:", int((TR["Tail_length"] > 0).sum()))
print("TRs after minimum-length filter:", int((TR["N_monomers"] >= minimum_root_length).sum()))
print("TRs used for screening:", len(TR_FILTERED))
print("Monomer period range:", TR_FILTERED["Period"].min(), "-", TR_FILTERED["Period"].max())

if len(TR_FILTERED) != 2308:
    print("NOTE: the archived screening run used 2,308 TRs; this input/filter combination gives a different count.")

In [ ]:
# Prepare position sets for the two independent comparisons

positions_stop_apo = {}
positions_divide = {}
positions_apo = {}
positions_non_apo = {}

for root in roots:
    root_df = (
        lineage[lineage["Root"] == root]
        .sort_values("Position")
        .reset_index(drop=True)
    )
    biological_indices = [
        i for i, row in root_df.iterrows()
        if row["Position_type"] == "BIOLOGICAL"
    ]

    positions_stop_apo[root] = [
        i for i in biological_indices
        if root_df.loc[i, "State"] in {"STOP", "APO"}
    ]
    positions_divide[root] = [
        i for i in biological_indices
        if root_df.loc[i, "State"] == "DIVIDE"
    ]
    positions_apo[root] = [
        i for i in biological_indices
        if root_df.loc[i, "State"] == "APO"
    ]
    positions_non_apo[root] = [
        i for i in biological_indices
        if root_df.loc[i, "State"] != "APO"
    ]

assert sum(map(len, positions_stop_apo.values())) == 487
assert sum(map(len, positions_divide.values())) == 455
assert sum(map(len, positions_apo.values())) == 96
assert sum(map(len, positions_non_apo.values())) == 846

print("STOP + APO positions:", sum(map(len, positions_stop_apo.values())))
print("DIVIDE positions:", sum(map(len, positions_divide.values())))
print("APO positions:", sum(map(len, positions_apo.values())))
print("Non-APO positions:", sum(map(len, positions_non_apo.values())))

In [ ]:
# Reusable full TR scan with separate result and checkpoint files

from pathlib import Path

THRESHOLD = 0.50
WORKDIR = Path("/content")

def scan_tr_pairs(left_positions, right_positions, result_name, state_name):
    result_file = WORKDIR / result_name
    state_file = WORKDIR / state_name
    temp_state_file = WORKDIR / (state_file.stem + "_TEMP.json")

    columns = [
        "TR_ID", "TR_Index", "Chromosome",
        "Start", "End", "Period", "Root", "Offset"
    ]

    if state_file.exists():
        state = json.loads(state_file.read_text())
        start_index = int(state["next_index"])
        if start_index < 0 or start_index > len(TR_FILTERED):
            raise ValueError("Invalid checkpoint index.")
        promoted_ids = (
            set(pd.read_csv(result_file)["TR_ID"].astype(str))
            if result_file.exists() else set()
        )
        print("Resuming from TR index:", start_index)
    else:
        start_index = 0
        promoted_ids = set()
        if result_file.exists():
            result_file.unlink()
        print("Starting a new scan.")

    for tr_index in range(start_index, len(TR_FILTERED)):
        row = TR_FILTERED.iloc[tr_index]
        period = int(row["Period"])
        sequence = row["Complete_sequence"]

        matrix = np.frombuffer(
            sequence.encode("ascii"), dtype=np.uint8
        ).reshape(-1, period)

        tr_id = (
            f"TR_{row['Chromosome']}_{row['Start']}_{row['End']}"
        )
        promoted_rows = []

        for root in roots:
            left = left_positions[root]
            right = right_positions[root]
            if not left or not right:
                continue

            max_position = max(left + right)
            number_of_offsets = len(matrix) - max_position

            for offset in range(max(0, number_of_offsets)):
                passed = True

                for left_pos in left:
                    left_monomer = matrix[offset + left_pos]

                    for right_pos in right:
                        right_monomer = matrix[offset + right_pos]
                        identity = np.count_nonzero(
                            left_monomer == right_monomer
                        ) / period

                        if identity >= THRESHOLD:
                            passed = False
                            break

                    if not passed:
                        break

                if passed:
                    promoted_rows.append({
                        "TR_ID": tr_id,
                        "TR_Index": tr_index,
                        "Chromosome": row["Chromosome"],
                        "Start": row["Start"],
                        "End": row["End"],
                        "Period": period,
                        "Root": root,
                        "Offset": offset
                    })

        if promoted_rows:
            pd.DataFrame(promoted_rows, columns=columns).to_csv(
                result_file,
                mode="a",
                header=not result_file.exists(),
                index=False
            )
            promoted_ids.add(tr_id)

        temp_state_file.write_text(
            json.dumps({"next_index": tr_index + 1})
        )
        temp_state_file.replace(state_file)

        print(
            f"\rTR scanned: {tr_index + 1}/{len(TR_FILTERED)}"
            f" | promoted TRs: {len(promoted_ids)}",
            end="", flush=True
        )

    print("\nScan completed.")

    results = (
        pd.read_csv(result_file)
        if result_file.exists()
        else pd.DataFrame(columns=columns)
    )

    print("TRs scanned:", len(TR_FILTERED))
    print("Promoted TRs:", results["TR_ID"].nunique())
    print("Promoted windows:", len(results))
    print("Roots with promoted windows:", results["Root"].nunique())
    print("Results:", result_file)
    print("Checkpoint:", state_file)

    return results

## Scan 1 — STOP + APO vs DIVIDE

The left group contains all biological STOP and APO positions. The right group contains DIVIDE positions. Virtual NULL positions are excluded.

In [ ]:
RESULTS_STOP_DIVIDE = scan_tr_pairs(
    positions_stop_apo,
    positions_divide,
    "AB_TR_PROMOSSE.csv",
    "AB_TR_STATO.json"
)

SUMMARY_STOP_DIVIDE = (
    RESULTS_STOP_DIVIDE
    .groupby(
        ["Root", "TR_ID", "Chromosome", "Start", "End", "Period"],
        as_index=False
    )
    .agg(
        Number_of_offsets=("Offset", "count"),
        Promoted_offsets=("Offset", lambda values: ", ".join(map(str, sorted(values))))
    )
    .sort_values(["Root", "TR_ID"])
    .reset_index(drop=True)
)

print("Distinct root/TR combinations:", len(SUMMARY_STOP_DIVIDE))
display(SUMMARY_STOP_DIVIDE.head(30))

## Scan 2 — APO vs non-APO

This comparison is independent of Scan 1. APO positions are compared with all other biological positions (DIVIDE + STOP). Virtual NULL positions are excluded.

In [ ]:
RESULTS_APO = scan_tr_pairs(
    positions_apo,
    positions_non_apo,
    "AB_TR_APO_PROMOSSE.csv",
    "AB_TR_APO_STATO.json"
)

roots_with_apo_hits = set(RESULTS_APO["Root"])
roots_without_apo_hits = [root for root in roots if root not in roots_with_apo_hits]

print("Roots with promoted APO windows:", len(roots_with_apo_hits))
print("Roots without promoted APO windows:", len(roots_without_apo_hits))
print("Roots without APO matches:", roots_without_apo_hits)

SUMMARY_APO = (
    RESULTS_APO
    .groupby(
        ["Root", "TR_ID", "Chromosome", "Start", "End", "Period"],
        as_index=False
    )
    .agg(
        Number_of_offsets=("Offset", "count"),
        Promoted_offsets=("Offset", lambda values: ", ".join(map(str, sorted(values))))
    )
    .sort_values(["Root", "TR_ID"])
    .reset_index(drop=True)
)

display(SUMMARY_APO.head(30))